# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadYakout/FlyRank-Ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?


**Chosen lane: Lane 2 — Refresh / Content Opportunity Scoring**

The goal of this lane is to help a content or SEO reviewer decide which pages should be reviewed first for possible refresh, expansion, protection, pruning, or monitoring.

**Task type: Ranking / scoring.**

The question is fundamentally "Which pages should be reviewed first?" rather than simply "Will this page decline?" Therefore, I would produce a priority score for each content item and use that score to create a ranked review queue.

The unit of analysis is **one row per pseudonymized content item/page**. Each row contains page-level search, traffic, engagement, content, and freshness signals.

The output supports a human review decision: the reviewer can inspect higher-priority pages first and decide what action, if any, is appropriate.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the starter dataset
df = pd.read_csv("/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of content items:", len(df))
print("Number of clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Number of content items: 30000
Number of clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The eventual target should represent an **observed page outcome in a future time window**, after the information used for scoring is available.

For example, a future outcome could identify whether a page experiences a meaningful decline in search performance during a later measurement window. The exact threshold and future window should be defined before model training and must be separated from the feature window to avoid leakage.

For this Week 2 framing exercise, I will sketch the target column rather than claim that the starter dataset already contains a leakage-safe future target.

The existing `trend_direction` column is useful for understanding the starter data, but it should not be treated as a model feature. The FlyRank data guidance states that `trend_direction` and `trend_pct` are derived from the decline logic and therefore should not be used as features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric will be **Precision@K**.

The practical question is not whether the model predicts every page perfectly. The practical question is whether the top K pages in the ranked queue contain a useful number of pages that meet the predefined future outcome.

Precision@K is appropriate because a reviewer has limited time and can only investigate a limited number of pages. A useful ranking should therefore place a relatively high proportion of relevant pages near the top of the queue.

The value of K should be chosen based on the realistic review capacity of the workflow and defined before evaluating the final model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


The output is a ranked list of content items with a priority score.

A content or SEO reviewer can use the ranking to decide which pages to inspect first. After reviewing the evidence and page context, the reviewer may choose an action such as refresh, expansion, protection, pruning, or monitoring.

The score is therefore a **decision-support tool**, not an automatic content-change system. A high score should mean "review this page earlier," not "this page definitely needs a refresh."

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select the lane-relevant page-level columns
lane_slice = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
        "freshness_tier"
    ]
].copy()

# Show the actual unit of analysis
lane_slice.head(10)

,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,freshness_tier
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,0.76,10.6,187,20,0-30
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,0.05,20.3,445,25,0-30
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,0.09,36.5,141,20,0-30
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,0.49,6.2,463,22,0-30
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,0.13,44.0,263,14,0-30
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,0.03,8.5,147,20,0-30
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,0.00,7.0,90,20,0-30
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,0.06,21.2,445,22,0-30
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,68,0.09,46.0,90,20,0-30
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,3,0.16,4.9,257,104,91-180


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A simple rule could prioritize pages using one condition, such as pages with declining impressions or pages with high impressions and low CTR. However, the starter dataset contains multiple signals, including impressions, clicks, sessions, position, CTR, content age, freshness, engagement, and content characteristics.

These signals can interact in ways that are difficult to represent with one fixed rule. For example, a page with declining impressions may be more important to review when it also has substantial search exposure and an established ranking position.

ML is worth testing because it may combine multiple signals and produce a more useful ranking than a single hand-written rule. However, ML should only be kept if it performs better than a simple baseline using the predefined evaluation metric. If a fixed rule performs just as well, the simpler approach may be preferable.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sketch the structure of the future target.
# This is NOT a training target yet because we do not have
# a separate future outcome window in this starter snapshot.

target_sketch = lane_slice[
    ["content_id", "client_id"]
].copy()

target_sketch["future_outcome"] = pd.Series(
    pd.NA,
    index=target_sketch.index,
    dtype="Int64"
)

target_sketch.head(10)

,content_id,client_id,future_outcome
0,content_304f48230142,client_f369cb89fc,<NA>
1,content_a1fb4e703a9e,client_4e07408562,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,<NA>
3,content_331d6c4de07b,client_19581e27de,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,<NA>
5,content_d4084a4bc775,client_f369cb89fc,<NA>
6,content_9a34b442b552,client_8722616204,<NA>
7,content_a63219c6e95a,client_19581e27de,<NA>
8,content_5e6c160719bc,client_6208ef0f77,<NA>
9,content_c27558df2b0c,client_19581e27de,<NA>


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.